# Malawi 1997 - Recreate 1000 Bins / Data Andres Post Reunion

Using the new Malawi 1997 microdata, we rebuilt 1,000 population-weighted bins with two approaches: a direct weighted-binning method and an adapted “official-style” pipeline (Lorenz-based functions plus duplicate-households logic). Both methods produced the same mean welfare (about 4.8658), with essentially zero difference at displayed precision. In this run, the duplicate-households correction was not needed (iterations = 0), which is consistent with why both pipelines converged to the same average.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 120)
pd.set_option("display.float_format", lambda x: f"{x:,.6f}")

## 1) Locate input file

In [11]:
ROOT = Path('.').resolve()

# Find project root from current notebook location.
PROJECT_ROOT = next(
    (p for p in [ROOT, *ROOT.parents] if (p / '.git').exists() or (p / 'pyproject.toml').exists()),
    ROOT,
 )

candidates = [
    PROJECT_ROOT / 'MWI_1997_new.dta',
    PROJECT_ROOT / '01-input' / 'country' / 'MWI_1997_new.dta',
]

mwi_path = next((p for p in candidates if p.exists()), None)
if mwi_path is None:
    raise FileNotFoundError(
        f'MWI_1997.dta not found. Looked in: {candidates}'
    )

PROJECT_ROOT, mwi_path

(WindowsPath('C:/Users/wb661551/OneDrive - WBG/Desktop/Internship/Bottom Censoring/bindata_check'),
 WindowsPath('C:/Users/wb661551/OneDrive - WBG/Desktop/Internship/Bottom Censoring/bindata_check/MWI_1997_new.dta'))

## 2) Load and prepare microdata

In [12]:
mwi_raw = pd.read_stata(mwi_path, convert_categoricals=False)
required_cols = ['welfare', 'weight']
missing = [c for c in required_cols if c not in mwi_raw.columns]
if missing:
    raise ValueError(f'Missing required columns: {missing}')

mwi = mwi_raw[['welfare', 'weight']].dropna().copy()
mwi = mwi[mwi['weight'] > 0].copy()

# Convert to 2021 PPP/day when CPI and ICP are available.
if 'cpi2021' in mwi_raw.columns and 'icp2021' in mwi_raw.columns:
    cpi = mwi_raw['cpi2021'].dropna()
    icp = mwi_raw['icp2021'].dropna()
    if not cpi.empty and not icp.empty:
        mwi['welfare'] = mwi['welfare'] / (float(cpi.iloc[0]) * float(icp.iloc[0]) * 365.0)

mwi.shape

(20000, 2)

In [13]:
mwi_raw

,reporting_level,welfare,weight,cw,cwy,cwy2,cwylog,index
0,national,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0
1,national,0.280488,528.101750,528.101750,148.126459,41.547766,-671.334887,1
2,national,0.281465,528.101750,"1,056.203500",296.768854,83.385464,"-1,340.833547",2
3,national,0.282442,528.101750,"1,584.305250",445.927187,125.514104,"-2,008.502342",3
4,national,0.285885,528.101750,"2,112.407000",596.903599,168.676007,"-2,669.773062",4
...,...,...,...,...,...,...,...,...
19996,national,336.939745,528.101750,"10,559,922.593000","38,883,704.869508","626,630,421.460968","9,845,562.169015",19996
19997,national,368.647780,528.101750,"10,560,450.694750","39,078,388.407482","698,400,075.619084","9,848,683.166740",19997
19998,national,400.355816,528.101750,"10,560,978.796500","39,289,817.014323","783,046,747.951926","9,851,847.739211",19998
19999,national,432.063851,528.101750,"10,561,506.898250","39,517,990.690031","881,632,344.924876","9,855,052.563433",19999


## 3) Build weighted equal-population bins (n=1000)

In [14]:
def weighted_equal_bins(df, value_col, weight_col, n_bins=1000, tolerance=1e-6):
    d = df[[value_col, weight_col]].dropna().copy()
    d = d[d[weight_col] > 0].copy()
    d = d.sort_values(value_col).reset_index(drop=True)

    total_weight = d[weight_col].sum()
    if total_weight <= 0:
        raise ValueError('Total weight must be > 0.')

    bin_size = total_weight / n_bins

    rows = []
    cur_bin = 1
    cur_weight = 0.0

    for v, w in zip(d[value_col].to_numpy(), d[weight_col].to_numpy()):
        remaining = float(w)

        while remaining > 0 and cur_bin <= n_bins:
            room = bin_size - cur_weight
            if abs(room) < tolerance:
                cur_bin += 1
                cur_weight = 0.0
                continue

            take = min(remaining, room)
            rows.append({
                'bin': cur_bin,
                'welfare': float(v),
                'weight': float(take),
            })

            remaining -= take
            cur_weight += take

            if cur_weight >= bin_size - tolerance:
                cur_bin += 1
                cur_weight = 0.0

    expanded = pd.DataFrame(rows)
    expanded['wt_welfare'] = expanded['welfare'] * expanded['weight']

    b = expanded.groupby('bin', as_index=False).agg(
        avg_welfare=('wt_welfare', lambda s: s.sum() / expanded.loc[s.index, 'weight'].sum()),
        pop=('weight', 'sum'),
        quantile=('welfare', 'max'),
        welfare_total=('wt_welfare', 'sum'),
    )

    b['pop_share'] = b['pop'] / b['pop'].sum()
    b['welfare_share'] = b['welfare_total'] / b['welfare_total'].sum()
    b.loc[b['bin'] >= n_bins, 'quantile'] = np.nan

    return b[['bin', 'avg_welfare', 'pop', 'pop_share', 'welfare_share', 'quantile']]

bins_1000 = weighted_equal_bins(mwi, value_col='welfare', weight_col='weight', n_bins=1000)
bins_1000.head(), bins_1000.tail()

(   bin  avg_welfare           pop  pop_share  welfare_share  quantile
 0    1     0.313979 10,562.035000   0.001000       0.000065  0.342887
 1    2     0.353957 10,562.035000   0.001000       0.000073  0.378334
 2    3     0.393859 10,562.035000   0.001000       0.000081  0.402255
 3    4     0.416352 10,562.035000   0.001000       0.000086  0.426518
 4    5     0.435401 10,562.035000   0.001000       0.000089  0.451911,
       bin  avg_welfare           pop  pop_share  welfare_share  quantile
 995   996    46.885646 10,562.035000   0.001000       0.009636 51.277309
 996   997    56.900297 10,562.035000   0.001000       0.011694 63.635859
 997   998    70.545418 10,562.035000   0.001000       0.014498 74.270248
 998   999    82.738549 10,562.035000   0.001000       0.017004 92.668099
 999  1000 1,296.486318 10,562.035000   0.001000       0.266449       NaN)

## 4) Quality checks

In [15]:
direct_mean = float(np.average(mwi['welfare'], weights=mwi['weight']))
bins_mean = float(np.average(bins_1000['avg_welfare'], weights=bins_1000['pop']))

checks = pd.DataFrame([{
    'n_bins': int(bins_1000['bin'].nunique()),
    'sum_pop_share': float(bins_1000['pop_share'].sum()),
    'sum_welfare_share': float(bins_1000['welfare_share'].sum()),
    'direct_weighted_mean': direct_mean,
    'mean_from_1000_bins': bins_mean,
    'difference': bins_mean - direct_mean,
}])

checks

,n_bins,sum_pop_share,sum_welfare_share,direct_weighted_mean,mean_from_1000_bins,difference
0,1000,1.000000,1.000000,4.865801,4.865801,0.000000


## 5) Save output

In [16]:
out_dir = PROJECT_ROOT / 'work' / 'EDA'
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / 'MWI_1997_recreated_1000_bins.csv'
bins_1000.to_csv(out_path, index=False)
out_path

WindowsPath('C:/Users/wb661551/OneDrive - WBG/Desktop/Internship/Bottom Censoring/bindata_check/work/EDA/MWI_1997_recreated_1000_bins.csv')

## 6) Pipeline oficial adaptado (ejecutable)

Este bloque adapta la estructura `Codigo 2 + Codigo 3 + Codigo 1` para recalcular bins en este notebook.

- `Codigo 2`: funciones de binning/lorenz.
- `Codigo 3`: modulo duplicate_households para corregir no-monotonicidad.
- `Codigo 1`: runner principal que usa los dos bloques anteriores, recrea 1000 bins y compara medias.

In [17]:
# Codigo 2: functions (adaptado a Python)

def new_bins_official(welfare, weight, nbins=1000, tolerance=1e-6, ids=None):
    welfare = np.asarray(welfare, dtype=float)
    weight = np.asarray(weight, dtype=float)

    valid = ~np.isnan(welfare) & ~np.isnan(weight)
    welfare = welfare[valid]
    weight = weight[valid]

    if ids is None:
        ids = np.arange(len(welfare))
    else:
        ids = np.asarray(ids)[valid]

    order = np.argsort(welfare)
    welfare = welfare[order]
    weight = weight[order]
    ids = ids[order]

    total_weight = weight.sum()
    if total_weight <= 0:
        raise ValueError("Total weight must be > 0")

    bin_size = total_weight / nbins

    out_rows = []
    cur_bin = 1
    cur_weight = 0.0

    for id_i, w, wt in zip(ids, welfare, weight):
        remaining = float(wt)
        while remaining > 0 and cur_bin <= nbins:
            room = bin_size - cur_weight

            if abs(room) < tolerance:
                cur_bin += 1
                cur_weight = 0.0
                continue

            take = min(remaining, room)
            out_rows.append(
                {
                    "id": int(id_i),
                    "bin": int(cur_bin),
                    "weight": float(take),
                    "welfare": float(w),
                }
            )

            remaining -= take
            cur_weight += take

            if cur_weight >= bin_size - tolerance:
                cur_bin += 1
                cur_weight = 0.0

    return pd.DataFrame(out_rows)


def lorenz_table_official(df, nq=1000, tolerance=1e-6):
    d = df.copy()

    if "reporting_level" not in d.columns:
        d["reporting_level"] = "national"

    if d["reporting_level"].nunique() > 1:
        d2 = d.copy()
        d2["reporting_level"] = "national"
        d = pd.concat([d, d2], ignore_index=True)

    pieces = []
    for reporting_level, sub in d.groupby("reporting_level", observed=True):
        sub = sub.sort_values("welfare").reset_index(drop=True)
        sub["id"] = np.arange(len(sub))

        binned = new_bins_official(
            welfare=sub["welfare"].to_numpy(),
            weight=sub["weight"].to_numpy(),
            ids=sub["id"].to_numpy(),
            nbins=nq,
            tolerance=tolerance,
        )
        binned["reporting_level"] = reporting_level
        pieces.append(binned)

    expanded = pd.concat(pieces, ignore_index=True)
    expanded["wt_welfare"] = expanded["welfare"] * expanded["weight"]

    totals = (
        expanded.groupby("reporting_level", observed=True)
        .agg(
            tot_pop=("weight", "sum"),
            tot_wlf=("wt_welfare", "sum"),
        )
        .reset_index()
    )

    expanded = expanded.merge(totals, on="reporting_level", how="left")
    expanded["pop_share"] = expanded["weight"] / expanded["tot_pop"]
    expanded["welfare_share"] = expanded["wt_welfare"] / expanded["tot_wlf"]

    lt = (
        expanded.groupby(["reporting_level", "bin"], observed=True)
        .apply(
            lambda g: pd.Series(
                {
                    "avg_welfare": np.average(g["welfare"], weights=g["weight"]),
                    "pop_share": g["pop_share"].sum(),
                    "welfare_share": g["welfare_share"].sum(),
                    "quantile": g["welfare"].max(),
                    "pop": g["weight"].sum(),
                }
            )
        )
        .reset_index()
        .sort_values(["reporting_level", "bin"])
        .reset_index(drop=True)
    )

    return lt

In [18]:
# Codigo 3: duplicate_households (adaptado a Python)

def find_outliers_by_weight(df, weight_col="weight", threshold=2.5):
    out = df.copy()
    mean_w = out[weight_col].mean()
    sd_w = out[weight_col].std(ddof=1)
    out["is_outlier"] = out[weight_col] > (mean_w + threshold * sd_w)
    return out


def optimize_ratio(y, m):
    y = np.asarray(y, dtype=float)
    opt_x = np.maximum(m, np.sqrt(y))
    opt_x[opt_x > (y / 2)] = 1
    return np.round(y / opt_x).astype(int)


def duplicate_obs(df, weight_col="weight"):
    out = df.copy()
    min_w = out[weight_col].min()

    out["hhindex"] = np.arange(len(out))
    out["rep_count"] = optimize_ratio(out[weight_col].to_numpy(), min_w)
    out.loc[~out["is_outlier"], "rep_count"] = 1
    out["rep_count"] = out["rep_count"].astype(int)

    return out.loc[out.index.repeat(out["rep_count"])].copy()


def add_new_weights(df, weight_col="weight"):
    out = df.copy()
    mask = out["is_outlier"]
    out.loc[mask, weight_col] = out.loc[mask, weight_col] / out.loc[mask, "rep_count"]
    return out


def duplicate_households_official(
    df,
    weight_col="weight",
    threshold=2.5,
    i=0,
    li=5,
    super_limit=20,
    nq=1000,
):
    R = df.copy()

    if i == 0:
        lt0 = lorenz_table_official(R, nq=nq)
        welfare_share_bad = (
            lt0.groupby("reporting_level", observed=True)["welfare_share"]
            .apply(lambda s: (s.diff().dropna() < 0).any())
            .any()
        )
        if not welfare_share_bad:
            return R, lt0, {"welfare_share_OK": True, "threshold": threshold, "iterations": i}

    if i >= super_limit:
        lt = lorenz_table_official(R, nq=nq)
        return R, lt, {"welfare_share_OK": False, "threshold": threshold, "iterations": i}

    while True:
        i += 1

        ori_cols = R.columns.tolist()
        R = find_outliers_by_weight(R, weight_col=weight_col, threshold=threshold)
        R = duplicate_obs(R, weight_col=weight_col)
        R = add_new_weights(R, weight_col=weight_col)

        # clean_new_weights equivalent
        R = R[[c for c in ori_cols if c in R.columns]].copy()

        lt = lorenz_table_official(R, nq=nq)
        welfare_share_bad = (
            lt.groupby("reporting_level", observed=True)["welfare_share"]
            .apply(lambda s: (s.diff().dropna() < 0).any())
            .any()
        )

        if not welfare_share_bad:
            return R, lt, {"welfare_share_OK": True, "threshold": threshold, "iterations": i}

        if i >= super_limit:
            return R, lt, {"welfare_share_OK": False, "threshold": threshold, "iterations": i}

        if welfare_share_bad and i >= li and threshold > 0:
            threshold = max(threshold - 0.5, 0)
            li = li * 2

In [19]:
# Codigo 1: main runner (usando Codigo 2 + Codigo 3)

nq = 1000

# Build input dataframe expected by the official-style pipeline.
dt_main = mwi[["welfare", "weight"]].copy()
dt_main["reporting_level"] = "national"

# Run duplicate_households + lorenz_table pipeline.
dt_rep, lt_official_style, algorithm_info = duplicate_households_official(
    dt_main,
    weight_col="weight",
    threshold=2.5,
    i=0,
    li=5,
    super_limit=20,
    nq=nq,
)

# Censoring step as in main script.
lt_official_style.loc[lt_official_style["bin"] >= nq, "quantile"] = np.nan

# Create id as in main script (single-country notebook adaptation).
country_code = "MWI"
year = 1997
welfare_type = "unknown"
lt_official_style["id"] = f"{country_code}_{year}_{welfare_type}"

# Keep final column order similar to exported production shape.
cols = ["id", "reporting_level", "bin", "avg_welfare", "pop_share", "welfare_share", "quantile", "pop"]
lt_official_style = lt_official_style[cols].sort_values(["id", "reporting_level", "bin"]).reset_index(drop=True)

# Means comparison.
direct_mean_official_input = float(np.average(dt_main["welfare"], weights=dt_main["weight"]))
mean_from_official_style_bins = float(np.average(lt_official_style["avg_welfare"], weights=lt_official_style["pop"]))

comparison_official_style = pd.DataFrame([
    {
        "method": "Direct weighted mean (input)",
        "value": direct_mean_official_input,
    },
    {
        "method": "Mean from official-style 1000 bins",
        "value": mean_from_official_style_bins,
    },
    {
        "method": "Difference",
        "value": mean_from_official_style_bins - direct_mean_official_input,
    },
    {
        "method": "welfare_share_OK",
        "value": float(1 if algorithm_info["welfare_share_OK"] else 0),
    },
    {
        "method": "iterations",
        "value": float(algorithm_info["iterations"]),
    },
    {
        "method": "final_threshold",
        "value": float(algorithm_info["threshold"]),
    },
])

comparison_official_style

,method,value
0,Direct weighted mean (input),4.865801
1,Mean from official-style 1000 bins,4.865801
2,Difference,0.000000
3,welfare_share_OK,1.000000
4,iterations,0.000000
5,final_threshold,2.500000


In [20]:
# Save official-style output
out_official_path = out_dir / "MWI_1997_recreated_1000_bins_official_style.csv"
lt_official_style.to_csv(out_official_path, index=False)
out_official_path

WindowsPath('C:/Users/wb661551/OneDrive - WBG/Desktop/Internship/Bottom Censoring/bindata_check/work/EDA/MWI_1997_recreated_1000_bins_official_style.csv')